In [2]:
import numpy as np
from math import sqrt, atan2, pi

np.set_printoptions(
    precision=6,
    suppress=True,
    linewidth=160
)

# ============================================================
# 1. INPUT
# ============================================================

# Koordinatrekkefølge:
# [Northing, Easting, kjent?]
#
# Dette passer med:
# atan2(dY, dX)
#
# hvor:
# X = Northing
# Y = Easting

punkter = {
    "P111": [6615743.203, 599821.802, True],
    "NMB1": [6615708.451, 599809.075, True],

    # P112 skal parameterestimeres.
    # Fastmerkekoordinaten brukes som startverdi.
    "P112": [6615726.406, 599772.346, False],
}


# Alle tre punktene brukes som stasjoner
orienteringer = {
    "P111": 0,
    "P112": 1,
    "NMB1": 2,
}


# Bare P112 har ukjente koordinater
ukjente = {
    "P112": [3, 4],
}

addisjonskonstant_col = 5
antall_ukjente = 6

# [o_P111, o_P112, o_NMB1, dN_P112, dE_P112,addisjonskonstantenn]

# ============================================================
# RETNINGSOBSERVASJONER
# ============================================================

# Hver Hz-middelverdi er en obs 

retning_obs = [


    # ---------- P111 -> P112 ----------

    ["P111", "P112", 43.1038],
    ["P111", "P112", 43.1014],
    ["P111", "P112", 43.1005],
    ["P111", "P112", 43.0987],


    # ---------- P111 -> NMB1 ----------

    ["P111", "NMB1", 386.3027],
    ["P111", "NMB1", 386.3060],
    ["P111", "NMB1", 386.3036],
    ["P111", "NMB1", 386.3041],


    # ---------- P112 -> P111 ----------

    #["P112", "P111", 354.7944],
    ["P112", "P111", 354.8015],
    ["P112", "P111", 354.8037],
    ["P112", "P111", 354.8040],



    # ---------- P112 -> NMB1 ----------

    ["P112", "NMB1", 4.5993],
    #["P112", "NMB1", 4.5922],
    ["P112", "NMB1", 4.5995],
    ["P112", "NMB1", 4.6014],


    # ---------- NMB1 -> P112 ----------

    #["NMB1", "P112", 171.4277],
    ["NMB1", "P112", 171.4201],
    ["NMB1", "P112", 171.4202],
    ["NMB1", "P112", 171.4248],


    # ---------- NMB1 -> P111 ----------

    ["NMB1", "P111", 264.8323],
    ["NMB1", "P111", 264.8329],
    ["NMB1", "P111", 264.8347],
    ["NMB1", "P111", 264.8340],
]
# ============================================================
# AVSTANDSOBSERVASJONER
# ============================================================

avstand_obs = [

    # ---------- P111 -> P112 ----------

    ["P111", "P112", 52.3410, 103.0064],
    ["P111", "P112", 52.3405, 103.0069],
    ["P111", "P112", 52.3405, 103.0078],
    ["P111", "P112", 52.3405, 103.0063],


    # ---------- P111 -> NMB1 ----------

    ["P111", "NMB1", 37.2615, 106.6486],
    ["P111", "NMB1", 37.2615, 106.6486],
    ["P111", "NMB1", 37.2615, 106.6478],
    ["P111", "NMB1", 37.2615, 106.6468],


    # ---------- P112 -> P111 ----------

    ["P112", "P111", 52.3405, 96.9925],
    ["P112", "P111", 52.3405, 96.9956],
    ["P112", "P111", 52.3405, 96.9961],
    ["P112", "P111", 52.3405, 96.9940],


    # ---------- P112 -> NMB1 ----------

    ["P112", "NMB1", 40.9480, 102.1976],
    ["P112", "NMB1", 40.9480, 102.1883],
    ["P112", "NMB1", 40.9475, 102.1974],
    ["P112", "NMB1", 40.9475, 102.1893],


    # ---------- NMB1 -> P112 ----------

    ["NMB1", "P112", 40.9480, 97.8054],
    ["NMB1", "P112", 40.9480, 97.8077],
    ["NMB1", "P112", 40.9485, 97.8072],
    ["NMB1", "P112", 40.9485, 97.8062],


    # ---------- NMB1 -> P111 ----------

    ["NMB1", "P111", 37.2610, 93.3555],
    ["NMB1", "P111", 37.2610, 93.3565],
    ["NMB1", "P111", 37.2610, 93.3572],
    ["NMB1", "P111", 37.2615, 93.3567],
]


sigma_retning = 0.0005  # gon

def sigma_avstand(D):
    # D i meter
    return 0.003 + 0.003 * (D / 1000)


# ============================================================
# 2. HJELPEFUNKSJONER
# ============================================================

def normaliser_gon(v):
    # holder vinkler i området -200 til 200 gon
    while v > 200:
        v -= 400
    while v < -200:
        v += 400
    return v


def beregn_retning(fra_punkt, til_punkt):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA

    r = atan2(dY, dX) * 200 / pi

    if r < 0:
        r += 400

    return r


def beregn_avstand(fra_punkt, til_punkt):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA

    return sqrt(dX**2 + dY**2)

# ============================================================
# 3. STARTORIENTERINGER
# ============================================================
# Første retning fra hver stasjon brukes som referanse.
# Dette gir l = 0 for første retning fra hver stasjon.

orientering_start = {}

for fra_punkt in orienteringer:
    for obs in retning_obs:
        if obs[0] == fra_punkt:
            til_punkt = obs[1]
            obs_retning = obs[2]

            beregnet = beregn_retning(fra_punkt, til_punkt)

            orientering_start[fra_punkt] = normaliser_gon(
                beregnet - obs_retning
            )
            break


# ============================================================
# 4. OBSERVASJONSLIKNINGER
# ============================================================

def lag_retning_obs(fra_punkt, til_punkt, obs_retning):
    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA
    D = sqrt(dX**2 + dY**2)

    beregnet = beregn_retning(fra_punkt, til_punkt)
    orientering = orientering_start[fra_punkt]

    # Rapport-vennlig konvensjon:
    # l = obs + orientering - beregnet
    l = normaliser_gon(obs_retning + orientering - beregnet)

    A_rad = [0] * antall_ukjente

    # Viktig: -1 for å følge rapportens A-matrise
    A_rad[orienteringer[fra_punkt]] = -1

    # derivert for til-punkt
    if til_punkt in ukjente:
        colX, colY = ukjente[til_punkt]

        A_rad[colX] = -(dY / D**2) * 200 / pi
        A_rad[colY] =  (dX / D**2) * 200 / pi

    # derivert for fra-punkt
    if fra_punkt in ukjente:
        colX, colY = ukjente[fra_punkt]

        A_rad[colX] =  (dY / D**2) * 200 / pi
        A_rad[colY] = -(dX / D**2) * 200 / pi

    # relativ vekt, slik rapporten bruker
    p = 1

    return A_rad, l, p


def lag_avstand_obs(fra_punkt, til_punkt, D_skraa, Z_middel):

    XA, YA = punkter[fra_punkt][0], punkter[fra_punkt][1]
    XB, YB = punkter[til_punkt][0], punkter[til_punkt][1]

    dX = XB - XA
    dY = YB - YA

    D = sqrt(dX**2 + dY**2)

    # Zenit fra gon til radianer
    Z_rad = Z_middel * pi / 200

    # Observert skråavstand projisert til horisontalplanet
    D_obs_hor = D_skraa * np.sin(Z_rad)

    # l = observert - beregnet
    l = D_obs_hor - D

    A_rad = [0] * antall_ukjente

    # derivert for til-punkt
    if til_punkt in ukjente:
        colX, colY = ukjente[til_punkt]

        A_rad[colX] = dX / D
        A_rad[colY] = dY / D

    # derivert for fra-punkt
    if fra_punkt in ukjente:
        colX, colY = ukjente[fra_punkt]

        A_rad[colX] = -dX / D
        A_rad[colY] = -dY / D

    # Addisjonskonstant:
    #
    # D_sann = D_obs + a
    #
    # Horisontalt:
    # D_hor = (D_obs + a) * sin(Z)
    #
    # Derfor koeffisient = -sin(Z)
    A_rad[addisjonskonstant_col] = -np.sin(Z_rad)

    sigma = sigma_avstand(D)

    p = sigma_retning**2 / sigma**2

    return A_rad, l, p


# ============================================================
# 5. BYGG A, l OG P
# ============================================================

A_liste = []
l_liste = []
p_liste = []
navn_liste = []

for fra_punkt, til_punkt, obs_retning in retning_obs:
    A_rad, l_verdi, p_verdi = lag_retning_obs(
        fra_punkt,
        til_punkt,
        obs_retning
    )

    A_liste.append(A_rad)
    l_liste.append(l_verdi)
    p_liste.append(p_verdi)
    navn_liste.append(f"Retning {fra_punkt}->{til_punkt}")

for fra_punkt, til_punkt, D_skraa, Z_middel in avstand_obs:
    A_rad, l_verdi, p_verdi = lag_avstand_obs(
        fra_punkt,
        til_punkt,
        D_skraa,
        Z_middel
    )

    A_liste.append(A_rad)
    l_liste.append(l_verdi)
    p_liste.append(p_verdi)
    navn_liste.append(f"Avstand {fra_punkt}->{til_punkt}")

A = np.array(A_liste, dtype=float)
l = np.array(l_liste, dtype=float).reshape(-1, 1)
P = np.diag(p_liste)


# ============================================================
# 6. MINSTE KVADRATERS METODE
# ============================================================

N = A.T @ P @ A
h = A.T @ P @ l

dx = np.linalg.solve(N, h)

v = A @ dx - l

frihetsgrader = len(l) - antall_ukjente

s0_hat = sqrt((v.T @ P @ v)[0, 0] / frihetsgrader)

Qxx = np.linalg.inv(N)

std_parametre = s0_hat * np.sqrt(np.diag(Qxx)).reshape(-1, 1)


# ============================================================
# 7. PEN UTSKRIFT
# ============================================================

kolonnenavn = [
    "o_P111",
    "o_P112",
    "o_NMB1",
    "dN_P112",
    "dE_P112",
    "Add_konstant",
] #NB ENDRE PÅ EXAMENNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN

def print_matrix(navn, M, rader=None, kolonner=None, des=6):
    print("\n" + "=" * 80)
    print(navn)
    print("=" * 80)

    M = np.array(M)

    if M.ndim == 1:
        M = M.reshape(-1, 1)

    if kolonner is None:
        kolonner = [f"c{i+1}" for i in range(M.shape[1])]

    if rader is None:
        rader = [f"r{i+1}" for i in range(M.shape[0])]

    print("".ljust(24), end="")
    for k in kolonner:
        print(f"{k:>14}", end="")
    print()

    for navn_rad, row in zip(rader, M):
        print(f"{navn_rad:<24}", end="")
        for verdi in row:
            print(f"{verdi:>14.{des}f}", end="")
        print()


print("\nSTARTORIENTERINGER")
for punkt, verdi in orientering_start.items():
    print(f"{punkt}: {verdi:.6f} gon")

print_matrix("A-matrise", A, navn_liste, kolonnenavn, des=6)
print_matrix("l-vektor", l, navn_liste, ["l"], des=6)
print_matrix("P diagonal", np.diag(P), navn_liste, ["p"], des=8)

print_matrix("N = A.T P A", N, kolonnenavn, kolonnenavn, des=6)
print_matrix("h = A.T P l", h, kolonnenavn, ["h"], des=8)

print_matrix("Korreksjoner dx", dx, kolonnenavn, ["dx"], des=8)
print_matrix("Residualer v = A dx - l", v, navn_liste, ["v"], des=8)

print("\n" + "=" * 80)
print("STANDARDAVVIK TIL VEKTSENHETEN")
print("=" * 80)
print(f"s0_hat = {s0_hat:.8f}")

print_matrix("Qxx = N^-1", Qxx, kolonnenavn, kolonnenavn, des=6)
print_matrix("Standardavvik til parametere", std_parametre, kolonnenavn, ["std"], des=8)


print("\n" + "=" * 80)
print("NYE KOORDINATER")
print("=" * 80)

for punkt in ukjente:
    colX, colY = ukjente[punkt]

    X_gammel = punkter[punkt][0]
    Y_gammel = punkter[punkt][1]

    X_ny = X_gammel + dx[colX, 0]
    Y_ny = Y_gammel + dx[colY, 0]

    print(f"{punkt}:")
    print(f"  X gammel = {X_gammel:.6f}")
    print(f"  Y gammel = {Y_gammel:.6f}")
    print(f"  dX       = {dx[colX, 0]:.8f}")
    print(f"  dY       = {dx[colY, 0]:.8f}")
    print(f"  X ny     = {X_ny:.6f}")
    print(f"  Y ny     = {Y_ny:.6f}")


print("\n" + "=" * 80)
print("ORIENTERINGSKORREKSJONER")
print("=" * 80)

for punkt in orienteringer:
    colO = orienteringer[punkt]

    print(f"{punkt}: do = {dx[colO, 0]:.8f} gon")
print("\n" + "=" * 80)
print("ESTIMERT ADDISJONSKONSTANT aka Prismekonstant")
print("=" * 80)

a = dx[addisjonskonstant_col, 0]

print(f"a = {a:.6f} m")
print(f"a = {a * 1000:.2f} mm")


STARTORIENTERINGER
P111: -163.947463 gon
P112: 124.354837 gon
NMB1: 157.526270 gon

A-matrise
                                o_P111        o_P112        o_NMB1       dN_P112       dE_P112  Add_konstant
Retning P111->P112           -1.000000      0.000000      0.000000      1.154115     -0.391978      0.000000
Retning P111->P112           -1.000000      0.000000      0.000000      1.154115     -0.391978      0.000000
Retning P111->P112           -1.000000      0.000000      0.000000      1.154115     -0.391978      0.000000
Retning P111->P112           -1.000000      0.000000      0.000000      1.154115     -0.391978      0.000000
Retning P111->NMB1           -1.000000      0.000000      0.000000      0.000000      0.000000      0.000000
Retning P111->NMB1           -1.000000      0.000000      0.000000      0.000000      0.000000      0.000000
Retning P111->NMB1           -1.000000      0.000000      0.000000      0.000000      0.000000      0.000000
Retning P111->NMB1           -1.0